In [ ]:
import os
import sys
from tqdm import tqdm
import glob
import typing
import import_ipynb

import numpy as np
import pandas as pd
from collections import defaultdict
import librosa, opensmile

# Add current directory to path for imports
import os
current_dir = "/home2/ducvu/speech-processing-implement/codes"
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

import importlib
import config
from config import *

In [ ]:
def analyze_metadata(df_metadata_final:pd) -> dict:

    # Dict for saving result
    stat_result = defaultdict()

    diagnosis_group = df_metadata_final.diagnosis.unique()

    # Diagnosis distribution
    diagnosis_count = df_metadata_final.diagnosis.value_counts()
    print(f"All data: {len(df_metadata_final)} and diagnosis count: {diagnosis_count}")

    # Age distribution
    age_stat = defaultdict()
    for diagnosis in diagnosis_group:
        group_data = df_metadata_final[df_metadata_final.diagnosis == diagnosis].age
        mean_age = np.mean(group_data)
        std_age = np.std(group_data)

        age_stat[diagnosis] = {
            "Mean age": mean_age,
            "Std age": std_age,
        }

    age_stat["Overall"] = {
        "Mean age": np.mean(df_metadata_final.age),
        "Std age": np.std(df_metadata_final.age),
    }

    bin_range = (0, 50, 60, 70, 80, 100)
    bin_labels = ["0 - 50", "50 - 60", "60 - 70", "70 - 80", "80 - 100"]

    age_histograms = defaultdict()
    for diagnosis in diagnosis_group:
        group_data = df_metadata_final[df_metadata_final.diagnosis == diagnosis].age
        hist, _ = np.histogram(group_data, bins=bin_range)

        age_histograms[diagnosis] = dict(zip(bin_labels, hist))

    stat_result["age stat"] = age_stat
    stat_result["age histograms"] = age_histograms

    # Ethnicity Distribution in disease
    ethnic_diagnosis = defaultdict()
    ethnic_group = df_metadata_final.ethnicity.unique()
    for ethnic in ethnic_group:
        diagnosis_count = df_metadata_final[df_metadata_final.ethnicity == ethnic].diagnosis.value_counts()
        ethnic_diagnosis[ethnic] = diagnosis_count

    stat_result["Ethnic diagnosis"] = ethnic_diagnosis

    # Gender distribution
    gender_diagnosis = defaultdict()
    for diagnosis in diagnosis_group:
        gender_count = df_metadata_final[df_metadata_final.diagnosis == diagnosis].age.value_counts()
        gender_diagnosis[diagnosis] = gender_count

    stat_result["gender diagnosis"] = gender_diagnosis


    return stat_result

In [ ]:
# Hyperparameter
CV_SCORER = config.CV_SCORER
N_FOLDS = config.N_FOLDS

# Directory
DATA_PATH = config.DATA_PATH
FEATS_PATH = config.FEATS_PATH
RESULTS_PATH = config.RESULTS_PATH

# List acoustic feature
LIST_ACOUSTIC = config.LIST_ACOUSTIC

# List classifier and chosen
LIST_CLASSIFIER_NAME = config.LIST_CLASSIFIER_NAME
CLASSIFIER_CHOSEN = config.CLASSIFIER_CHOSEN

# List task chosen
TASK_CHOSEN = config.TASK_CHOSEN

# List class type chosen
CLASS_TYPE_CHOSEN = config.CLASS_TYPE_CHOSEN

# Way for classifying
WAY_CLASSIFICATION = config.WAY_CLASSIFICATION

# Mapping label
LABEL_MAP = config.LABEL_MAP

In [ ]:
df_metadata_final = pd.read_csv(f"{DATA_PATH}/metadata.csv")


In [ ]:
df_metadata_final

,Unnamed: 0,participant_id,age,gender,diagnosis,ethnicity
0,0,participant_001,62,F,HC,White-British
1,1,participant_002,58,M,HC,White-British
2,2,participant_003,69,F,HC,Asian
3,3,participant_004,64,M,HC,Other
4,4,participant_005,71,F,HC,White-British
5,5,participant_006,78,M,Dementia,White-British
6,6,participant_007,82,F,Dementia,White-British
7,7,participant_008,75,M,Dementia,Asian
8,8,participant_009,80,F,Dementia,Mixed
9,9,participant_010,77,M,Dementia,White-British


In [ ]:
stat_result = analyze_metadata(df_metadata_final)
print(stat_result.items())

All data: 10 and diagnosis count: diagnosis
HC          5
Dementia    5
Name: count, dtype: int64
dict_items([('age stat', defaultdict(None, {'HC': {'Mean age': np.float64(64.8), 'Std age': np.float64(4.707440918375928)}, 'Dementia': {'Mean age': np.float64(78.4), 'Std age': np.float64(2.416609194718914)}, 'Overall': {'Mean age': np.float64(71.6), 'Std age': np.float64(7.761443164772902)}})), ('age histograms', defaultdict(None, {'HC': {'0 - 50': np.int64(0), '50 - 60': np.int64(1), '60 - 70': np.int64(3), '70 - 80': np.int64(1), '80 - 100': np.int64(0)}, 'Dementia': {'0 - 50': np.int64(0), '50 - 60': np.int64(0), '60 - 70': np.int64(0), '70 - 80': np.int64(3), '80 - 100': np.int64(2)}})), ('Ethnic diagnosis', defaultdict(None, {'White-British': diagnosis
HC          3
Dementia    3
Name: count, dtype: int64, 'Asian': diagnosis
HC          1
Dementia    1
Name: count, dtype: int64, 'Other': diagnosis
HC    1
Name: count, dtype: int64, 'Mixed': diagnosis
Dementia    1
Name: count, dtype